# AWS and SageMaker

In this lesson, we'll learn about the importance of being able to write production-quality code to make the models you've trained usable, and the leading cloud environments that make this possible!


## Objectives 

- Explain why cloud computing and putting models into production is important to data scientists 


## Data Science Skills and the Job Market

You're almost done with your studies, and will soon be ready to begin the job hunt. Since this is one of the most important topics a Data Scientist can know, we've elected to leave it until the very end, so that it'll be fresh in your mind going into your capstone project and the start of your career -- **_putting models into production_**. 

As you start to look at job postings, you probably notice that a "Data Scientist" job can mean many, many different things. In some companies, it means a data analyst or a DBA focused on databases or data pipelines. In others, it means someone with a scientific mindset skilled with running A/B tests. Yet others may be highly specialized machine learning roles in areas like NLP, Computer Vision, or Deep Learning -- and these roles may break down further into specializations focused on either research or implementation. As a Junior Data Scientist entering the workforce, it's most likely that you'll land in a generalist role, spending the first few years of your career working on various tasks that focus on all of these areas at least a little bit. Specialization happens later in your career. Out of the gate, the best thing that you can be is a strong generalist, with the demonstrated ability to contribute to many different sorts of projects that might be expected of a Data Science team. 

Over the course of your studies, you've picked up many different skills in many different domain areas that will allow you to contribute to data science projects in a professional environment. However, when it comes to the job market, not all Data Science skills are created equal. While different data scientists or recruiters may rank these skills differently based on their own experiences or needs, one thing most agree on is that the ability to **_productionize a model_** is both very valuable and very rare when it comes to Junior Data Scientists. This presents a massive opportunity for you -- if you can become proficient in productionizing the models you've created (and demonstrate this proficiency in your portfolio of projects!), you become a much more interesting candidate for any role.  

## Productionizing Models as a Career Skill

At large companies such as Google and Facebook, Data Scientists typically run experiments and train models until they have found a solution that works. Once they have trained a validated the model, they typically then hand off productionization of the model to **_Machine Learning Engineers_**. Whereas the Data Scientist creates the basic prototype, the Machine Learning Engineer's job is to put that model into a production system in a performant and maintainable manner. Whereas Data Scientists at large companies focus on the "big picture" by finding solutions to business problems, Machine Learning Engineers focus on the details, implementing the solutions created by the data scientists in the best way possible. Data Scientists focus more on analytics and statistics, whereas Machine Learning Engineers will have a stronger command of backend engineering, data structures and algorithms, and software engineering overall. The following diagram lays out the relationship between different technical roles well:

<img src='assets/data_science_and_machine_learning_engineering/new-venn-diagram.png' height=80% width=80%>

As you can see from the overlap between _Data Scientist_ and _Machine Learning Engineer_ , there is still significant overlap between the two -- a Data Scientist should be able to productionize a basic machine learning model, just as a Machine Learning Engineer should able to train a model, validate results, and deal with overfitting. 

### Being a 'Scrappy' Data Scientist

Many junior data scientists have at least one or two areas where they have significant holes in their knowledge. There are many paths into data science, and many junior data scientists are from backgrounds that have overlap with certain parts of data science. For instance, it's not uncommon for bioinformatics professionals or statisticians to rebrand themselves as a data scientist to take advantage of the higher salary in this field. While their backgrounds may give them a very strong understanding of analytics, scientific experimentation, or understanding how machine learning models work, these professionals often have little exposure to engineering, and thus can create and train models, but aren't able to put them into production so that the company can actually use them. Similarly, many junior data scientists on the job market have nothing more than a bachelor's degree in computer science and a passing understanding of machine learning -- in this case, productionizing a model is easy, but they may lack depth of understanding when it comes to the model itself. This isn't an insurmountable problem -- large companies almost always have some role where a candidate's skills can be useful, and they can invest in training the employee and skilling them up in areas where they're a bit weak. 

With small and medium-sized companies, this is a much bigger problem. Data Scientists in smaller organizations are expected to be a bit more independent, and will likely have to "wear more hats". For a data science role at a startup, it's a common expectation for their data scientists to handle every part of a data science project. This means starting by interviewing key stakeholders and identifying the problem to be solved, followed by rapidly prototyping a solution until you've trained/tuned/validated a model that meets your standards, followed by actually putting that model in production!  This means that it's very important to be 'scrappy', and be able to handle anything that's thrown at you as a data scientist. Smaller companies often don't have the funds or the infrastructure for a separate Machine Learning Engineering team to handle the details of implementation. In this respect, being able to productionize a machine learning model is the most practical, useful skill you can have in your Data Science toolbox. For all but the largest companies, it doesn't matter how great you are at training models -- until you put that model into production so that the rest of the organization can actually _use_ it, it might as well not exist! 

The TL;DR here is quite simple:

1. Many data scientists don't know how to put machine learning models into production.  
2. Putting a model into production is a mandatory skill for data scientists at most small to medium-sized companies.
3. Being able to productionize models will make you a much more attractive candidate to employers, and give you a competitive advantage!


## Data Science and Cloud Computing

We've established that being able to productionize machine learning models is a valuable skill -- so how do we do it? This answer has changed in recent years thanks to cloud computing platforms such as **_Amazon Web Services_**. A decade ago, productionizing a machine learning model would have meant building your own web server with something like [Flask](http://flask.pocoo.org/) or [Django](https://www.djangoproject.com/) and hosting somewhere, just like you would with any web app. However, the creation of cloud computing platforms changed things in a big way, and data science is no exception. Now, we don't even need to worry about things like server code -- instead, we can use preexisting services from AWS that were created specifically to simplify the process of productionizing machine learning solutions!

For the remainder of this section, we're going to dig deep into all the amazing tools AWS provides, and learn how we can use them to make you more effective data scientists!

---

In [ ]:
%%sh

# The name of our algorithm
algorithm_name=sagemaker-keras-text-classification

cd container

chmod +x sagemaker_keras_text_classification/train
chmod +x sagemaker_keras_text_classification/serve

account=$(aws sts get-caller-identity --query Account --output text)

# Get the region defined in the current configuration (default to us-west-2 if none defined)
region=$(aws configure get region)
region=${region:-us-west-2}

fullname="${account}.dkr.ecr.${region}.amazonaws.com/${algorithm_name}:latest"

# If the repository doesn't exist in ECR, create it.

aws ecr describe-repositories --repository-names "${algorithm_name}" > /dev/null 2>&1

if [ $? -ne 0 ]
then
    aws ecr create-repository --repository-name "${algorithm_name}" > /dev/null
fi

# Get the login command from ECR and execute it directly
$(aws ecr get-login --region ${region} --no-include-email)

# Build the docker image locally with the image name and then push it to ECR
# with the full name.

# On a SageMaker Notebook Instance, the docker daemon may need to be restarted in order
# to detect your network configuration correctly.  (This is a known issue.)
if [ -d "/home/ec2-user/SageMaker" ]; then
  sudo service docker restart
fi

docker build  -t ${algorithm_name} .
docker tag ${algorithm_name} ${fullname}

docker push ${fullname}

## Step 2: Setting Up the Environment

Once we've created the container, we'll need to set up the environment. The cell below contains more boilerplate code, which is used to handle a couple sticking points in order to set up the environment.

In [ ]:
# S3 prefix
prefix = 'sagemaker-keras-text-classification'

# Define IAM role
import boto3
import re

import os
import numpy as np
import pandas as pd
from sagemaker import get_execution_role

role = get_execution_role()

## Step 3: Creating the Session

Now that we've created the container and set up our environment, the next step is to create a SageMaker session.

In [ ]:
import sagemaker as sage
from time import gmtime, strftime

sess = sage.Session()

## Step 4: Upload the Data for Training

Steps 4 and 5 are where you'll add the code unique to your project.In this step, make sure have a folder called `'data'` that contains the data you'll be working with. The actual structure of the data is up to you, as you'll be the one consuming it to train your model in step 5.

In [ ]:
WORK_DIRECTORY = 'data'

data_location = sess.upload_data(WORK_DIRECTORY, key_prefix=prefix)

## Step 5: Fitting the Model

This is the part where you'll do the brunt of the work. You'll train your own model on the data you uploaded in the previous step. Note that in the sample code below, the first 3 lines are boilerplate code. The actual creation and training of the model happen on the last two lines of code, where `tree` is instantiated and used. 

**_NOTE_**: You may have noticed that the code in the cell below uses an `Estimator` from `sage` (which is just an alias we set for `sagemaker` up above), the SageMaker library for python, rather than a model from scikit-learn. The `sagemaker` library contains a massive amount of useful models that we can use directly. Under the hood, the `sagemaker` library wraps in the same open-source frameworks such as scikit-learn, Keras, and TensorFlow that you're used to using. The code below is an example from AWS of how to use one of their `Estimator` objects for training. If you read the output of the cell when you run everything, you'll notice that much of it is warning messages or other printouts from sklearn and keras!

For more information on the models and other tools included in the aws sagemaker library, check out [Amazon SageMaker Python SDK Documentation](https://sagemaker.readthedocs.io/en/stable/)!

In [ ]:
account = sess.boto_session.client('sts').get_caller_identity()['Account']
region = sess.boto_session.region_name
image = '{}.dkr.ecr.{}.amazonaws.com/sagemaker-keras-text-classification'.format(account, region)

tree = sage.estimator.Estimator(image,
                       role, 1, 'ml.c5.2xlarge',
                       output_path="s3://{}/output".format(sess.default_bucket()),
                       sagemaker_session=sess)

tree.fit(data_location)

## Step 6: Deploying the Model

This is where the magic happens -- we have a trained model, and now we need to actually **_deploy_** it to the AWS cloud! Notice how during this step, we include a `json_serializer` -- this is so that the model can serialize and deserialize data as needed when taking data in as input. 

Running the cell below will create an endpoint for your trained model.

In [ ]:
from sagemaker.predictor import json_serializer
predictor = tree.deploy(1, 'ml.t2.medium', serializer=json_serializer)

## Step 7: Cleanup  (IMPORTANT!)

As a final step for this exercise, be sure to run the following line of code to delete your endpoint! Although you are running this lab on the free tier, you don't want to leave it running, because that is how costs can accrue. Run the cell below to delete your endpoint.

In [ ]:
sess.delete_endpoint(predictor.endpoint)

## Step 8: Deactivate Everything in AWS

In AWS, you pay for usage. This means that anything left running is being used.  While the AWS Free Tier we've signed up for allows us to do small things for free for prototyping or learning, leaving some things running may take us past the usage limits for the AWS Free Tier. In order to avoid getting charged, you'll need to do the following steps: 

### 8.1: Deactivate the notebook in Sagemaker

First, you'll need to deactivate your notebook in SageMaker. When you enter the SageMaker platform, you'll always see the number of open notebooks you have up and running highlighted in green under the 'Recent Activity' section. 

<img src='assets/productionizing_models_with_sagemaker/create-notebook-7.png'>

To deactivate a running notebook, select it and then go to the 'Actions' tab and select stop. Stopping the notebook instance will take a minute or two. You'll know it's done when you see the 'Status' column for the highlighted notebook change from 'InService' to 'Stopped'. 

<img src='assets/productionizing_models_with_sagemaker/create-notebook-8.png'>

### 8.2: Keep an Eye on Cost Explorer

As you've seen from this lab, getting a handle on all the different services in AWS and how they interact with one another can be a bit daunting until you have some experience. It's very important that you don't leave services running when you aren't using them, because you will be charged for that. If you want to make sure that you haven't left anything running, the easiest thing to do is to check the 'Costs Explorer' page inside AWS. You can find this by searching for 'AWS Cost Explorer' in the search bar on the main page for the AWS Console. This service will show you what your usage is for everything that you can be charged for. It's quite intuitive and easy to use, and should make it easy to see if you are accruing charges because you left something running that you didn't realize. If you left something running that you aren't aware of, you'll see it here -- once you've noticed it, just navigate to the service in question and deactivate it.

---

In this lesson, we'll get set up to use **_Amazon Web Services_**, and then get to know our way around the platform before digging into AWS SageMaker in the next lesson. 

<img src='assets/the_aws_ecosystem/awscloud.svg'>


## Objectives 

- Set up an AWS account and explore the Amazon Resource Center 
- Explain what the "regions" are in AWS and why it is important to choose the right one 


## Getting Started

Before we can begin exploring everything AWS has to offer, we'll need to create an account on the platform. To do this, start by following this link to [Amazon Web Services](https://aws.amazon.com/). While you're there, you may want to take the time to bookmark it -- chances are this is a website you'll use frequently in your career as a Data Scientist!

### Will This Cost Money?

Although you will need a credit card to register for AWS, working through this section will not cost any money. AWS provides a free tier for learning and prototyping on the platform -- this is the tier we'll use for everything going forward. As long as you correctly register for the free tier, this will not cost you any money. 

### Register Your Email

Begin by clicking the "Sign Up" button in the top right-hand corner of the page. 

<img src='assets/the_aws_ecosystem/aws-1.png'>

Next, create an account by adding your email and password. You'll also need to set an **_AWS Account Name_**. 

<img src='assets/the_aws_ecosystem/aws-2.png'>

On the next screen, enter your contact information. **_Make sure you set your account type to 'Personal'!_** 

<img src='assets/the_aws_ecosystem/aws-3.png'>

This next page is especially important -- be sure to select the **_Basic Plan_**! As a reminder, you will be asked to enter a credit card number during the next few steps. Although we will only be making use of the free tier of services for AWS, be aware that you will still need to enter a credit card number in order to complete the registration process. 

<img src='assets/the_aws_ecosystem/aws-4.png'>

Now that you're all signed up, click the "Sign in to the Console" button to actually enter the AWS Console. 

<img src='assets/the_aws_ecosystem/aws-5.png'>

Alright, you've now created an AWS Account! Let's take a look around. 

## The AWS Console

Now that you're signed in, you'll see the **_AWS Console_**. This is your "home screen" for AWS -- it allows you to quickly navigate through the thousands of services offered on AWS to find what you need. The easiest way to find what you need is the "Find Services" search bar at the top of the body of the page. 

<img src='assets/the_aws_ecosystem/aws-6.png'>

You can also click the "See All Services" dropdown to see a full list of services you can use in AWS. There are **a ton** of services, but don't let yourself get overwhelmed -- you'll probably never end up using the vast majority of these, as only a few apply to the work of a data scientist. 

## Use Cases for Data Scientists

We've now created an account for AWS, so that we can take advantage of the Cloud. As data scientists, we'll find that a cloud computing service like AWS is very helpful in a number of ways. Aside from productionizing the model as a whole, the most important thing the cloud enables data scientists to do is to train much, much larger models by distributing training across entire clusters of servers. Without cloud computing, it would be impossible to train some of the larger deep learning models that exist today. The ability to distribute training of a neural network across a GPU allowed AI researchers to create massive models in a reasonable amount of time by creating a server cluster full of hundreds of GPUs. While this works, building a server like this is cost prohibitive to all but major companies and universities. Thankfully, services like AWS allow us to rent time on these servers per minute, making distributed training available for anybody at extremely cheap prices, paying only for what we use. AWS provides other great uses for data scientists beyond speedy training times -- it also plays a major part with databases. In your job as a data scientist, the databases you connect to in order to get your data will almost certainly be stored on AWS, or a competitor cloud platform. AWS servers also allow for companies to make use of big data frameworks such as Hadoop or Spark across a cluster of servers. 

## Using the Amazon Resource Center

As platforms go, you won't find many with more options than AWS. It has an amazing amount of offerings, with more getting added all the time. While AWS is great for basic use cases like hosting a server or a website, it also has all kinds of different offerings in areas such as Databases, Machine Learning, Data Analytics and other areas useful to Data Scientists. It's not possible for us to cover how to use every service in AWS in this section -- but luckily, we don't need to, because Amazon already has! The [Getting Started Resource Center](https://aws.amazon.com/getting-started/) contains a ton of awesome tutorials, demonstrations, and sample projects for just about everything you would ever want to know about any service on AWS. We **_strongly recommend_** bookmarking this page, as the tutorials they offer are very high quality, and free!

<img src='assets/the_aws_ecosystem/aws-7.png'>


## A Note On Regions

Before we move onto digging into **_AWS SageMaker_** in the next lesson, it's worth taking a moment to explain "Regions" and what they have to do with AWS. AWS has data centers all over the world, and they are **not** interchangeable when it comes to your projects. Click on the "Region" tab in the top right corner of the navigation bar, and you should see a dropdown of all the different data centers you can choose from. It is **_very important_** that you always choose the same region to connect to with your projects. Each region is its own unique data center, and anything you do on in that region is only in that region. One of the most common mistakes newcomers to AWS make is thinking they've lost their project because they are connected to a different data center and don't realize it. We'll remind you of this again later, but it can't hurt to say it twice: always make sure you're connected to the correct data center! This goes doubly for when you're creating a new project.